# Batch PDF Processing with RayJob on OpenShift AI

This notebook demonstrates how to process thousands of PDF documents at scale using
[Ray Data](https://docs.ray.io/en/latest/data/data.html) and
[Docling](https://github.com/DS4SD/docling) on Red Hat OpenShift AI (RHOAI),
using the **RayJob** custom resource via the CodeFlare SDK.

## RayJob vs RayCluster

The companion notebook (`ray-docling-batch-processing.ipynb`) uses the CodeFlare SDK
to create a **RayCluster** and then submits a job to it separately. This notebook
uses a **RayJob** instead, which bundles the cluster specification and the job
entrypoint into a single Kubernetes resource.

| Aspect | RayCluster notebook | RayJob notebook (this one) |
|---|---|---|
| Cluster lifecycle | Manual (create, then delete) | Automatic (created on submit, deleted on completion) |
| Job submission | Separate step via Ray Job Client | Part of the RayJob CR |
| SDK class | `Cluster` + `ClusterConfiguration` | `RayJob` + `ManagedClusterConfig` |
| Best for | Interactive / iterative development | Production pipelines, GitOps, CI/CD |

## What does this notebook do?

It converts a large collection of PDF files into **Markdown** and **JSON** representations
by distributing the work across a Ray cluster. Each PDF is parsed by Docling, which
extracts text, tables, and layout information with high fidelity.

## Architecture

```
Notebook (this file)
  │
  ├─ CodeFlare SDK ───▶ Creates RayJob CR on OpenShift
  │     │
  │     ▼
  │   ┌──────────────────────────────────────┐
  │   │  KubeRay Operator                     │
  │   │  Creates RayCluster + submits job     │
  │   └──────────────────────────────────────┘
  │                       ┌───────────────────────────────┐
  │                       │  Head Pod (no actors)          │
  │                       │  GCS • Dashboard • Job Server  │
  │                       │  num-cpus: "0"                 │
  │                       └───────────────────────────────┘
  │                       ┌───────────────────────────────┐
  │                       │  Worker Pod 1                  │
  │                       │    Actor 1 ─▶ [subprocess]    │
  │                       │    Actor 2 ─▶ [subprocess]    │──┐
  │                       │    Actor 3 ─▶ [subprocess]    │  │
  │                       └───────────────────────────────┘  │
  │                       ┌───────────────────────────────┐  │
  │                       │  Worker Pod 2 – N             │  │  PVC
  │                       │    ...                        │──┤  /mnt/data
  │                       └───────────────────────────────┘  │
  │                                                          │
  └─ Results: Markdown + JSON files ◀─────────────────────┘
```

Each actor runs Docling inside a **dedicated subprocess**. If a PDF causes the converter
to hang, the subprocess is killed after a configurable timeout and a fresh one is started
automatically.

## Prerequisites

- An OpenShift cluster with **RHOAI** (Red Hat OpenShift AI) installed
- The **KubeRay** operator enabled in your namespace
- The **CodeFlare SDK** (`codeflare-sdk`) Python package installed in your notebook environment
- A **ReadWriteMany PVC** (e.g. `data-pvc`) with your input PDFs uploaded
- A container image with Docling and Ray pre-installed (e.g. `quay.io/cathaloconnor/docling-ray:latest`)

---
## Step 1 — Import Dependencies

In [ ]:
import json
import os
import subprocess

from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes import client as kclient
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)
from kubernetes.client.rest import ApiException
from ray.runtime_env import RuntimeEnv

---
## Step 2 — Connect to OpenShift

Replace the `TOKEN` and `API_URL` values below with your own OpenShift credentials.
You can obtain a token from the OpenShift web console under **Copy login command**.

In [ ]:
# ---- REPLACE THESE WITH YOUR VALUES ----
TOKEN = "sha256~your-token-here"
API_URL = "https://api.your-cluster.example.com:443"
NAMESPACE = "ray-docling"

In [ ]:
!oc login --token={TOKEN} --server={API_URL}

---
## Step 3 — Configure Pipeline Parameters

This section defines all tunable parameters for the cluster and the processing job.

### Key equations

| Parameter | Formula |
|---|---|
| Schedulable CPUs per worker | `worker_cpus - 2` (2 reserved for Ray overhead) |
| Actors per worker | `schedulable_cpus // cpus_per_actor` |
| Max actors (cluster-wide) | `num_workers × actors_per_worker` |
| Total blocks | `max_actors × repartition_factor` |
| Files per block | `num_files / total_blocks` |
| Memory per actor | `(worker_memory - object_store) / actors_per_worker` |

> **Tip:** Use the companion `helper/configure.py` calculator to explore different
> cluster configurations before deploying.

In [ ]:
# ---------------------------------------------------------------------------
# Cluster sizing
# ---------------------------------------------------------------------------
NUM_WORKERS = 4  # Number of Ray worker pods
WORKER_CPUS = 15  # CPUs per worker pod (must fit on your nodes)
WORKER_MEMORY_GB = 32  # Memory per worker pod (GB)
HEAD_CPUS = 4  # CPUs for the head pod (no actors run here)
HEAD_MEMORY_GB = 8  # Memory for the head pod (GB)

# ---------------------------------------------------------------------------
# Actor configuration
#   With 15 CPUs per worker, 2 are reserved for Ray overhead (raylet,
#   object store), leaving 13 schedulable.  At 4 CPUs/actor that gives
#   3 actors/worker × 4 workers = 12 actors total.
# ---------------------------------------------------------------------------
CPUS_PER_ACTOR = 4  # CPUs allocated to each Docling actor
MIN_ACTORS = 4  # Minimum actor pool size
MAX_ACTORS = 12  # Maximum actor pool size
BATCH_SIZE = 4  # Files per batch sent to each actor call

# ---------------------------------------------------------------------------
# Data partitioning
#   A higher repartition factor creates more (smaller) blocks, which reduces
#   tail latency from straggler documents.  With 12 actors × 40 = 480 blocks
#   and 10,000 files, each block has ~21 files.
# ---------------------------------------------------------------------------
NUM_FILES = 10000  # How many PDFs to process (0 = all)
REPARTITION_FACTOR = 40  # blocks = max_actors × this value

# ---------------------------------------------------------------------------
# Timeout and fault tolerance
# ---------------------------------------------------------------------------
TIMEOUT_SECONDS = 600  # Per-file timeout (kill converter if exceeded)
MAX_ERRORED_BLOCKS = 100  # Block-level errors tolerated before job abort

# ---------------------------------------------------------------------------
# Storage paths
# ---------------------------------------------------------------------------
PVC_NAME = "data-pvc"
PVC_MOUNT_PATH = "/mnt/data"
INPUT_PATH = "input/pdfs/10000"  # Relative to PVC_MOUNT_PATH
OUTPUT_PATH = "output"  # Relative to PVC_MOUNT_PATH

# Container image with Ray + Docling pre-installed
IMAGE = "quay.io/cathaloconnor/docling-ray:latest"

# RayJob settings
RAYJOB_NAME = "ray-docling-processor"
TTL_AFTER_FINISHED = 600  # Seconds to keep the RayJob CR after completion

In [ ]:
# Derived values (used in the RayJob spec)
SCHEDULABLE_CPUS = WORKER_CPUS - 2

print(
    f"Workers:            {NUM_WORKERS} x ({WORKER_CPUS} CPUs, {WORKER_MEMORY_GB} GB)"
)
print(f"Schedulable CPUs:   {SCHEDULABLE_CPUS} per worker")
print(f"Actors per worker:  {SCHEDULABLE_CPUS // CPUS_PER_ACTOR}")
print(f"Max actors:         {MAX_ACTORS}")
print(f"Total blocks:       {MAX_ACTORS * REPARTITION_FACTOR}")
print(f"Files per block:    ~{NUM_FILES // (MAX_ACTORS * REPARTITION_FACTOR)}")
print(
    f"Memory per actor:   ~{WORKER_MEMORY_GB * 0.9 / (SCHEDULABLE_CPUS // CPUS_PER_ACTOR):.1f} GB"
)

---
## Step 4 — Verify the PVC

Before creating the cluster, confirm that the shared PVC exists and uses
`ReadWriteMany` access mode. All worker pods will mount this PVC to read input
PDFs and write output files.

In [ ]:
def verify_pvc():
    """Check that the shared PVC exists and has the correct access mode."""
    configuration = kclient.Configuration()
    configuration.host = API_URL
    configuration.verify_ssl = True
    configuration.api_key["authorization"] = f"Bearer {TOKEN}"

    v1 = kclient.CoreV1Api(kclient.ApiClient(configuration))

    print(f"Checking PVC '{PVC_NAME}' in namespace '{NAMESPACE}'...")
    try:
        pvc = v1.read_namespaced_persistent_volume_claim(
            name=PVC_NAME, namespace=NAMESPACE
        )
        print(f"  Status:      {pvc.status.phase}")
        access_modes = pvc.spec.access_modes or []
        print(f"  Access mode: {access_modes[0] if access_modes else 'N/A'}")
        if access_modes and access_modes[0] != "ReadWriteMany":
            print("  WARNING: PVC should use ReadWriteMany for concurrent writes.")
    except ApiException as e:
        if e.status == 404:
            print(f"  ERROR: PVC '{PVC_NAME}' not found. Create it before continuing.")
        else:
            print(f"  ERROR: {e}")


verify_pvc()

---
## Step 5 — Review the Processing Script

The processing logic lives in `ray_data_process.py` (shared by both the
RayCluster and RayJob notebooks). For the RayJob approach, this script is
embedded in a ConfigMap and mounted into the head pod.

### How it works

1. **`_converter_worker`** — A persistent subprocess that initialises the Docling
   `DocumentConverter` once, then processes files one at a time from a
   `multiprocessing.Queue`.  Running Docling in a subprocess means we can
   **hard-kill** it if a PDF causes it to hang, without affecting the Ray actor.

2. **`DoclingProcessor`** — A lightweight Ray Data actor that sends file paths to
   the converter subprocess and waits for results with a timeout.  If no result
   arrives within `FILE_TIMEOUT` seconds, it kills the subprocess and starts a
   fresh one.

3. **`ray_data_process`** — The driver function that builds a `ray.data.Dataset`
   from the PDF file paths, repartitions it into many small blocks, and applies
   `DoclingProcessor` via `map_batches` with an `ActorPoolStrategy`.

### Why subprocess isolation instead of signal-based timeout?

Ray Data runs the actor's `__call__` method inside a **worker thread**, not the
main thread.  Python's `signal.SIGALRM` can only be received by the main thread,
so it would interrupt Ray's internal `future.result()` call rather than the
conversion code — crashing the entire job.  The subprocess + queue approach avoids
this entirely: `queue.Empty` is a normal Python exception that is handled gracefully
within the actor.

In [ ]:
# Verify the processing script exists

script_path = os.path.join(os.getcwd(), "ray_data_process.py")
assert os.path.exists(script_path), f"Processing script not found at {script_path}"
print(f"Processing script: {script_path}")
print(f"Size: {os.path.getsize(script_path):,} bytes")

---
## Step 6 — Create and Submit the RayJob

The CodeFlare SDK's `RayJob` class bundles the **cluster specification** and the
**job entrypoint** into a single RayJob custom resource. When submitted, the
KubeRay operator:

1. Creates a RayCluster with the head and worker specs
2. Waits for the cluster to become ready
3. Submits the job entrypoint to the cluster
4. Deletes the cluster automatically when the job completes

After submission, we apply a JSON patch to set `rayStartParams` — the same
approach used in the RayCluster notebook.

> **Note:** The CodeFlare SDK does not currently expose `rayStartParams`
> directly in `ManagedClusterConfig`, which is why we patch as a separate step.

In [ ]:
# Volume mount shared by head and workers
shared_mount = V1VolumeMount(PVC_MOUNT_PATH, name="shared-data")
data_volume = V1Volume(
    name="shared-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)

cluster_config = ManagedClusterConfig(
    # Head pod — runs GCS, dashboard, job server (no actors)
    head_cpu_requests=HEAD_CPUS,
    head_cpu_limits=HEAD_CPUS,
    head_memory_requests=HEAD_MEMORY_GB,
    head_memory_limits=HEAD_MEMORY_GB,
    # Worker pods
    num_workers=NUM_WORKERS,
    worker_cpu_requests=WORKER_CPUS,
    worker_cpu_limits=WORKER_CPUS,
    worker_memory_requests=WORKER_MEMORY_GB,
    worker_memory_limits=WORKER_MEMORY_GB,
    # Shared storage and container image
    volume_mounts=[shared_mount],
    volumes=[data_volume],
    image=IMAGE,
    # Platform annotations
    annotations={
        "odh.ray.io/secure-trusted-network": "false",
    },
    # Environment variables applied to all pods
    envs={
        # Keep object store small — we pass file paths, not file bytes
        "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION": "0.1",
        # Disable Ray TLS (ODH cert SANs don't cover pod IPs)
        "RAY_USE_TLS": "0",
    },
)

print("ManagedClusterConfig created.")

In [ ]:
job = RayJob(
    job_name=RAYJOB_NAME,
    entrypoint="python ray_data_process.py",
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    runtime_env=RuntimeEnv(
        working_dir=".",
        pip=["opencv-python-headless", "pypdfium2", "orjson"],
        env_vars={
            # Storage
            "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
            "INPUT_PATH": INPUT_PATH,
            "OUTPUT_PATH": OUTPUT_PATH,
            # Actor pool
            "NUM_FILES": str(NUM_FILES),
            "MIN_ACTORS": str(MIN_ACTORS),
            "MAX_ACTORS": str(MAX_ACTORS),
            "CPUS_PER_ACTOR": str(CPUS_PER_ACTOR),
            "BATCH_SIZE": str(BATCH_SIZE),
            "REPARTITION_FACTOR": str(REPARTITION_FACTOR),
            # Fault tolerance
            "FILE_TIMEOUT": str(TIMEOUT_SECONDS),
            "MAX_ERRORED_BLOCKS": str(MAX_ERRORED_BLOCKS),
            # Ray settings
            "RAY_DATA_ENABLE_RICH_PROGRESS_BARS": "true",
            "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION": "0.1",
            # Cache directories (shared across pods via PVC)
            "HF_HOME": f"{PVC_MOUNT_PATH}/.cache/huggingface",
            "XDG_CACHE_HOME": f"{PVC_MOUNT_PATH}/.cache/xdg",
            "DOCLING_CACHE_DIR": f"{PVC_MOUNT_PATH}/.cache/docling",
        },
    ),
    ttl_seconds_after_finished=TTL_AFTER_FINISHED,
)

print(f"RayJob '{RAYJOB_NAME}' configured.")

In [ ]:
# Submit the RayJob — this creates the RayJob CR and the KubeRay operator
# will create the RayCluster and submit the job automatically.
job.submit()
print(f"RayJob '{RAYJOB_NAME}' submitted.")

# Patch rayStartParams (not yet exposed by ManagedClusterConfig)
schedulable_cpus = WORKER_CPUS - 2
patch = [
    {
        "op": "replace",
        "path": "/spec/rayClusterSpec/enableInTreeAutoscaling",
        "value": True,
    },
    {
        "op": "add",
        "path": "/spec/rayClusterSpec/headGroupSpec/rayStartParams/num-cpus",
        "value": "0",
    },
    {
        "op": "add",
        "path": "/spec/rayClusterSpec/workerGroupSpecs/0/rayStartParams/num-cpus",
        "value": str(schedulable_cpus),
    },
]

print(f"Patching RayJob: head num-cpus=0, workers num-cpus={schedulable_cpus}...")
subprocess.run(
    [
        "oc",
        "patch",
        "rayjob",
        RAYJOB_NAME,
        "-n",
        NAMESPACE,
        "--type",
        "json",
        "-p",
        json.dumps(patch),
    ],
    check=True,
)
print("Patch applied.")

---
## Step 7 — Monitor the Job

You can track progress in several ways:

1. **CodeFlare SDK** — `job.status()` shows the overall job state.
2. **Pod status** — `oc get pods` shows which head/worker pods are running.
3. **Ray Dashboard** — accessible via the head pod's dashboard port.
4. **Job logs** — fetch from the head pod for the performance report.

In [ ]:
# Check job status via CodeFlare SDK
job.status()

In [ ]:
# List all pods created by this RayJob
!oc get pods -n {NAMESPACE} -l ray.io/cluster={RAYJOB_NAME} --sort-by=.metadata.creationTimestamp

In [ ]:
# Get the Ray Dashboard URL (route or port-forward)
!oc get route -n {NAMESPACE} -l ray.io/cluster={RAYJOB_NAME} -o jsonpath='{.items[0].spec.host}' 2>/dev/null && echo "" || echo "No route found. Use port-forward instead:"
print(f"  oc port-forward svc/{RAYJOB_NAME}-head-svc 8265:8265 -n {NAMESPACE}")
print("  Then open http://localhost:8265")

In [ ]:
# Fetch job logs from the head pod (run after the job completes for the performance report)
!oc logs -n {NAMESPACE} -l ray.io/cluster={RAYJOB_NAME},ray.io/node-type=head --tail=200

In [ ]:
# Full RayJob status (detailed)
!oc get rayjob {RAYJOB_NAME} -n {NAMESPACE} -o yaml | grep -A 20 'status:'

---
## Step 8 — Cleanup

The KubeRay operator automatically deletes the RayCluster when the job completes.
The RayJob CR itself is cleaned up after `ttl_seconds_after_finished` (default:
600s).

To manually clean up before that:

In [ ]:
# Delete the RayJob (also deletes the associated RayCluster)
# job.delete()

---
## Appendix — Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| Worker pods stuck in **Pending** | Pod CPU/memory requests exceed node capacity | Reduce `WORKER_CPUS` to fit your nodes (check `oc describe node`) |
| RayJob stays in **Initializing** | Cluster pods not ready | Check `oc get pods` and `oc describe pod` for scheduling issues |
| Job aborts with **UserCodeException** | `signal.SIGALRM` timeout (old approach) | Use this notebook's subprocess-based timeout instead |
| Single actor holds the job for hours | A PDF hangs Docling's C library | `FILE_TIMEOUT` will kill the subprocess and move on |
| **OOM-killed** worker pods | Not enough memory per actor | Increase `WORKER_MEMORY_GB` or reduce actors per worker by increasing `CPUS_PER_ACTOR` |
| Job finishes fast but only 1 actor used | `num-cpus` patch not applied | Re-run Step 6; check `oc get rayjob -o yaml` for `rayStartParams` |
| SSL errors in logs (`ssl_transport_security.cc`) | ODH cert SANs don't cover pod IPs | Benign — `RAY_USE_TLS=0` is set in the spec; processing is unaffected |
| Files per block too high (>50) | Low `REPARTITION_FACTOR` | Increase to 40+ to create smaller blocks and reduce straggler risk |

### Resource planning quick reference

```
schedulable_cpus  = WORKER_CPUS - 2
actors_per_worker = schedulable_cpus // CPUS_PER_ACTOR
max_actors        = NUM_WORKERS × actors_per_worker
memory_per_actor  = (WORKER_MEMORY_GB × 0.9) / actors_per_worker
total_blocks      = max_actors × REPARTITION_FACTOR
files_per_block   = NUM_FILES / total_blocks
```

For best results, ensure `memory_per_actor >= 4 GB` and `files_per_block` is
between 5 and 50.  Use the companion `helper/configure.py` tool to explore configurations
interactively:

```bash
python helper/configure.py --num-files 10000 --num-workers 4 \
  --worker-cpus 15 --worker-memory 32 --cpus-per-actor 4
```

### Re-running the job

RayJob names must be unique. To re-run the job, either:

1. Delete the existing RayJob first: `job.delete()`
2. Or change `RAYJOB_NAME` to a new value (e.g. append a timestamp)